# 🔬 XRD Analysis of Starch Polymorphs
### FBMFOR — Food Fraud Analysis | University of Reading

This notebook guides you through:
1. Loading XRD diffractogram data from the course dataset
2. Visualising patterns as a **waterfall plot**
3. Identifying and assigning **polymorph peaks** (A-type or B-type)
4. Estimating **relative crystallinity**
5. Identifying an **unknown starch sample** by peak matching

---
> **Fraud context:** Starch is a common adulterant in spices, cocoa, and dairy powders.  
> XRD polymorph fingerprinting can identify starch botanical origin and detect substitution.

## 1 · Setup

In [1]:
# ============================================================
# SETUP — INSTALL AND IMPORT LIBRARIES
# ============================================================
# All libraries used here are pre-installed in Google Colab.
# If running locally, install with:  pip install numpy pandas matplotlib scipy

# --- Core numerical and data libraries ---
import numpy as np                       # Array operations and numerical computation
import pandas as pd                      # DataFrames for tabular data handling

# --- Plotting ---
import matplotlib.pyplot as plt          # Static publication-quality figures
import matplotlib.patches as mpatches    # Custom legend patches (coloured rectangles)
from matplotlib.lines import Line2D      # Custom legend line handles

# --- Signal processing ---
from scipy.signal import find_peaks, savgol_filter
# find_peaks   : local-maxima detection algorithm (used in Sections 6 & 8)
# savgol_filter: Savitzky-Golay smoothing (used in Section 4)

from scipy.interpolate import interp1d   # 1D interpolation (used when resampling spectra)
import os, io                            # File-system and in-memory byte-stream utilities

# --- Colab file upload helper ---
# Imported inside a try/except so the notebook also works in a local Jupyter session.
# IN_COLAB is used throughout to switch between upload and local-file code paths.
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("\u2713 Libraries loaded")
print(f"  Running in Google Colab: {IN_COLAB}")

✓ Libraries loaded
  Running in Google Colab: False


## 2 · Reference Peak Library

Literature 2θ positions (Cu Kα, λ = 1.5406 Å) for the main starch polymorphs.
Edit this cell if you want to add new starch types or adjust positions.

In [4]:
# ============================================================
# REFERENCE PEAK LIBRARY
# ============================================================
# Literature 2θ positions (Cu Kα, λ = 1.5406 Å) for the two starch
# polymorph families present in the dataset.
#
# The crystal structure of starch is determined by the double-helix
# packing of amylopectin chains in the semicrystalline lamellae:
#
#   A-type (monoclinic unit cell, space group B2):
#     Characteristic of cereal starches (Wheat, Corn, Rice).
#     The double helices are packed more tightly and with less water
#     than B-type. Diagnostic peaks appear at 15.0°, 17.0°, 17.9°, 
#     22.9°.
#
#   B-type (hexagonal unit cell, space group P6₁):
#     Characteristic of tuber and root starches (Potato, Tapioca).
#     Wider channels in the lattice accommodate more water molecules.
#     The hallmark low-angle peak at ~5.6° arises from the larger
#     d-spacing of the (100) reflection in the hexagonal lattice.
#     5.6, 15.1, 17.2, 19.7, 22.2, 24.0, 26.4
#
# Source: Dome K, Podgorbunskikh E, Bychkov A, Lomovsky O. 
# Changes in the Crystallinity Degree of Starch Having Different 
# Types of Crystal Structure after Mechanical Pretreatment. 
# Polymers (Basel). 2020 Mar 12;12(3):641. 
# doi: 10.3390/polym12030641. PMID: 32178224; PMCID: PMC7183072.

REFERENCE_PEAKS = {
    # A-type (cereal starches) — monoclinic unit cell
    # Peaks at 15.3°/17.1°/18.0° form the characteristic A-type triplet.
    "Wheat":   {"peaks": [15.0, 17.0, 17.9, 22.9],             "polymorph": "A-type"},
    "Corn":    {"peaks": [15.0, 17.0, 17.9, 22.9],             "polymorph": "A-type"},

    # B-type (tuber / root starches) — hexagonal unit cell
    # The 5.6° peak is the most diagnostic: it is absent in A-type starches.
    "Potato":  {"peaks": [5.6, 15.1, 17.2, 19.7, 22.2, 24.0, 26.4], "polymorph": "B-type"},
    "Tapioca": {"peaks": [5.6, 15.1, 17.2, 19.7, 22.2, 24.0, 26.4], "polymorph": "B-type"},
}

# --- Colour maps ---
# One colour per polymorph family, used for reference-peak markers and the legend.
POLYMORPH_COLOURS = {
    "A-type":  "#C0392B",   # Red family — cereal starches
    "B-type":  "#1F618D",   # Blue family — tuber/root starches
}

# One colour per individual starch, used for diffractogram lines and fill.
# A-type starches share red tones; B-type starches share blue tones;
# Unknown is purple to remain visually neutral until identified.
STARCH_COLOURS = {
    "Wheat":   "#922B21",   # Dark red
    "Corn":    "#E74C3C",   # Bright red
    "Potato":  "#1A5276",   # Dark blue
    "Tapioca": "#2E86C1",   # Bright blue
    "Unknown": "#6C3483",   # Purple (polymorph unknown a priori)
}

print("\u2713 Reference library loaded")
for name, info in REFERENCE_PEAKS.items():
    print(f"  {name:12s} ({info['polymorph']:7s})  peaks: {info['peaks']}")

✓ Reference library loaded
  Wheat        (A-type )  peaks: [15.0, 17.0, 17.9, 22.9]
  Corn         (A-type )  peaks: [15.0, 17.0, 17.9, 22.9]
  Potato       (B-type )  peaks: [5.6, 15.1, 17.2, 19.7, 22.2, 24.0, 26.4]
  Tapioca      (B-type )  peaks: [5.6, 15.1, 17.2, 19.7, 22.2, 24.0, 26.4]


## 3 · Load Data

Data are loaded directly from the course GitHub repository.
If you are running this notebook locally with `data/xrd.csv` present, it will be used instead.

The expected format is a merged CSV with columns `TwoTheta, Counts, Starch`.
All five starch samples (Corn, Potato, Tapioca, Wheat, Unknown) are in a single file.

In [3]:
# ============================================================
# LOAD XRD DATA
# ============================================================
# The dataset (xrd.csv) contains diffractograms for five starch types:
#   Corn, Potato, Tapioca, Wheat, Unknown
# Each diffractogram was measured from 5° to 65° 2θ using Cu Kα radiation.
# The file is a long-format CSV with three columns:
#   TwoTheta  — 2θ angle in degrees
#   Counts    — raw detector counts (arbitrary intensity units)
#   Starch    — sample label
#
# Loading is attempted in order of convenience:
#   1. GitHub raw URL  — works in Colab with no extra steps
#   2. Colab file upload — fallback if GitHub is unreachable
#   3. Local file path  — for running the notebook on your own machine

GITHUB_URL = (
    "https://raw.githubusercontent.com/ggkuhnle/fbmfor-foodfraud/main/data/xrd.csv"
)

df_raw = None  # Will hold the raw DataFrame once loaded

# --- 1: Try GitHub URL ---
# pandas.read_csv() accepts a URL directly. This is the simplest route in Colab
# as it requires no file upload and always fetches the current course dataset.
try:
    df_raw = pd.read_csv(GITHUB_URL)
    print(f"\u2713 Loaded data from GitHub ({df_raw.shape[0]} rows)")
except Exception as e:
    print(f"  GitHub load failed ({e}); trying upload/local …")

# --- 2: Colab file upload ---
# If the GitHub fetch failed (e.g., network restricted), offer a manual upload.
if df_raw is None and IN_COLAB:
    print("\U0001f4c2 Upload xrd.csv:")
    uploaded = colab_files.upload()
    fname = list(uploaded.keys())[0]
    df_raw = pd.read_csv(io.BytesIO(uploaded[fname]))
    print(f"\u2713 Loaded from upload ({df_raw.shape[0]} rows)")

# --- 3: Local file fallback ---
# When running in Jupyter outside Colab, look for the file in the standard
# repository layout (data/xrd.csv) or the current working directory.
if df_raw is None:
    local_paths = ["data/xrd.csv", "xrd.csv"]
    for p in local_paths:
        if os.path.exists(p):
            df_raw = pd.read_csv(p)
            print(f"\u2713 Loaded from {p} ({df_raw.shape[0]} rows)")
            break

if df_raw is None:
    raise FileNotFoundError(
        "Could not load xrd.csv. Check your internet connection or upload the file."
    )

# --- Validate and parse ---
# Strip any accidental whitespace from column names (common in exported CSVs).
df_raw.columns = [c.strip() for c in df_raw.columns]
required = {"TwoTheta", "Counts", "Starch"}
if not required.issubset(df_raw.columns):
    raise ValueError(f"Missing columns. Found: {list(df_raw.columns)}, need: {required}")

# Build spectra_dict: { starch_label: (two_theta_array, counts_array) }
# groupby() splits the long-format DataFrame by sample label.
# sort_values() ensures 2θ runs in ascending order for all subsequent analysis.
spectra_dict = {}
for starch, grp in df_raw.groupby("Starch"):
    grp = grp.sort_values("TwoTheta")
    spectra_dict[starch.strip()] = (grp["TwoTheta"].values, grp["Counts"].values)
    print(f"  {starch}: {len(grp)} points, "
          f"2θ {grp['TwoTheta'].min():.1f}–{grp['TwoTheta'].max():.1f}°")

print(f"\n\u2713 {len(spectra_dict)} starch pattern(s): {list(spectra_dict.keys())}")

  GitHub load failed (HTTP Error 404: Not Found); trying upload/local …


FileNotFoundError: Could not load xrd.csv. Check your internet connection or upload the file.

## 4 · Preprocessing

Optional smoothing (Savitzky–Golay filter).  
Set `SMOOTH = False` to skip.

In [ ]:
# ============================================================
# PREPROCESSING — SMOOTHING AND NORMALISATION
# ============================================================
# Two lightweight preprocessing steps prepare the raw counts for analysis:
#
#   1. Savitzky-Golay (SG) smoothing
#      Fits a low-degree polynomial to a sliding window of points and replaces
#      the central value with the fitted value. This reduces Poisson counting
#      noise without distorting peak positions or widths, provided the window
#      is narrow relative to the peak width.
#
#      Note: here we use the zero-derivative (smoothing) mode. The chemometrics
#      notebook uses SG *derivatives* to resolve overlapping FTIR bands. That
#      approach is not used here because the broad amorphous hump in XRD is
#      analytically important for crystallinity estimation and would be suppressed
#      by differentiation.
#
#   2. Min-max normalisation to 0-100
#      Scales each diffractogram independently so its minimum maps to 0 and its
#      maximum maps to 100. This allows direct visual comparison of patterns
#      regardless of differences in raw counting statistics between samples
#      (e.g., due to different packing density in the XRD sample holder).

# ── PARAMETERS — modify and re-run to experiment ───────────────────────────────────
SMOOTH       = True    # Set to False to skip smoothing entirely.
SG_WINDOW    = 11      # Window length in data points (must be odd).
                       # At a typical step size of 0.02°/point, 11 points ≈ 0.22°.
                       # Increase (e.g., to 21) for noisier data;
                       # decrease if peaks appear rounded or flattened.
SG_POLYORDER = 3       # Polynomial degree. 3 gives good peak shape preservation.
                       # Must be less than SG_WINDOW.
# ──────────────────────────────────────────────────────────────────────────────


def normalise(counts):
    """
    Min-max normalise a 1D intensity array to the range [0, 100].

    Scaling to 0-100 rather than 0-1 keeps values at a human-readable scale
    when printed or displayed in the waterfall plot.

    Parameters:
        counts : 1D numpy array — raw or smoothed detector counts

    Returns:
        1D numpy array — rescaled intensities in [0, 100]
    """
    lo, hi = counts.min(), counts.max()
    if hi > lo:
        return (counts - lo) / (hi - lo) * 100
    else:
        return np.zeros_like(counts)  # Flat pattern: return zeros rather than NaN


# --- Apply smoothing and normalisation to every sample ---
# processed stores a tuple (two_theta, smoothed_counts, normalised_counts).
# Both the raw-scale smoothed counts (used later for crystallinity integration)
# and the 0-100 normalised version (used for plotting and peak detection)
# are retained so that each downstream step can use the appropriate scale.
processed = {}
for starch, (tt, ct) in spectra_dict.items():
    if SMOOTH:
        # Guard against the window exceeding the array length (rare with real data,
        # but important for robustness with very short test arrays).
        window = min(SG_WINDOW, len(ct) - 1)
        window = window if window % 2 == 1 else window - 1  # Enforce odd length
        ct_smooth = savgol_filter(ct, window_length=window, polyorder=SG_POLYORDER)
        ct_smooth = np.maximum(ct_smooth, 0)  # Clip negative artefacts near zero baseline
    else:
        ct_smooth = ct.copy()
    processed[starch] = (tt, ct_smooth, normalise(ct_smooth))

print(f"\u2713 Preprocessing complete ({'SG smoothed' if SMOOTH else 'no smoothing'})")
print(f"  Samples: {list(processed.keys())}")

## 5 · Waterfall Plot

Each diffractogram is offset vertically for clarity.  
Peak reference positions are annotated per starch type.

In [ ]:
# ============================================================
# WATERFALL PLOT
# ============================================================
# A waterfall plot stacks each diffractogram vertically, offset by a fixed
# amount, so all patterns can be compared side-by-side without overlap.
# This is the standard way to present a series of powder diffractograms.
#
# Two visual aids help pattern recognition:
#   - Dashed vertical grid lines mark the diagnostic 2θ positions for both
#     polymorph families, making it easy to see which samples share peaks.
#   - Small arrows mark the expected reference peak positions for each labelled
#     starch. The Unknown sample has no reference arrows because its identity
#     is what we are trying to determine.

# ── PARAMETERS — modify and re-run ────────────────────────────────────────────────
X_MIN       = 5.0     # Lower 2θ limit for the plot (degrees).
                      # 5° captures the B-type low-angle peak at 5.6°.
X_MAX       = 40.0    # Upper 2θ limit. All diagnostic starch peaks fall below 25°;
                      # extending to 40° confirms the absence of unexpected reflections.
OFFSET_STEP = 115     # Vertical offset between patterns (normalised intensity units).
                      # Increase if patterns overlap; decrease for a more compact view.
SHOW_GRID   = True    # Show dashed vertical lines at diagnostic 2θ positions.
SAVE_FIGURE = True    # Save the figure as a PNG file.
FIGURE_DPI  = 200     # Resolution in dots per inch (150 for screen, 300 for print).
FIGURE_NAME = "xrd_waterfall.png"
# ──────────────────────────────────────────────────────────────────────────────

starch_order = list(processed.keys())
n = len(starch_order)

fig, ax = plt.subplots(figsize=(12, 2.2 * n + 1.5))
ax.set_facecolor("white")

# --- Diagnostic grid lines ---
# Draw thin dashed verticals at the reference peak positions for both
# polymorph types. Spanning all patterns lets students see at a glance
# which samples produce a reflection at each diagnostic position.
if SHOW_GRID:
    for gp in [5.6, 14.4, 15.3, 17.1, 17.2, 18.0, 19.8, 22.2, 23.0, 24.0]:
        if X_MIN <= gp <= X_MAX:
            ax.axvline(gp, color="grey", lw=0.5, ls="--", alpha=0.35, zorder=0)

# --- Draw each diffractogram ---
for i, starch in enumerate(starch_order):
    tt, ct_raw, ct_norm = processed[starch]

    # Restrict to the plotted 2θ window
    mask = (tt >= X_MIN) & (tt <= X_MAX)
    tt_m, ct_m = tt[mask], ct_norm[mask]

    base = i * OFFSET_STEP   # Baseline elevation for this pattern
    y    = ct_m + base        # Shift the pattern upward by the cumulative offset

    col = STARCH_COLOURS.get(starch, "#555555")

    # Semi-transparent fill under the curve (aids visual separation of patterns)
    ax.fill_between(tt_m, base, y, alpha=0.18, color=col, zorder=i + 1)
    ax.plot(tt_m, y, color=col, lw=1.5, zorder=i + 2)  # Solid pattern line
    ax.axhline(base, color="grey", lw=0.4, ls="-", alpha=0.5, zorder=0)  # Baseline rule

    # --- Sample label in the right margin ---
    # Show the polymorph type for reference starches; leave it blank for Unknown.
    poly = REFERENCE_PEAKS.get(starch, {}).get("polymorph", "")
    label_text = f"{starch}\n({poly})" if poly else starch
    ax.text(X_MAX + 0.5, base + 48,
            label_text, fontsize=9, color=col, fontweight="bold",
            va="center", ha="left", zorder=i + 3)

    # --- Reference peak arrows ---
    # Draw upward-pointing arrows at the expected 2θ positions for each reference
    # starch. Unknown is not in REFERENCE_PEAKS so no arrows appear for it,
    # prompting students to read the peak positions directly from the pattern.
    ref = REFERENCE_PEAKS.get(starch)
    if ref:
        pcol = POLYMORPH_COLOURS[ref["polymorph"]]
        for p in ref["peaks"]:
            if X_MIN <= p <= X_MAX:
                idx = np.argmin(np.abs(tt_m - p))  # Nearest data point to reference
                py  = y[idx]
                ax.annotate("", xy=(p, py + 3), xytext=(p, py + 13),
                            arrowprops=dict(arrowstyle="-|>", color=pcol,
                                            lw=1.1, mutation_scale=7),
                            zorder=i + 4)
                ax.text(p, py + 14, f"{p}°",
                        fontsize=5.5, color=pcol, fontweight="bold",
                        ha="center", va="bottom", rotation=90, zorder=i + 5)

# --- Axes and labels ---
ax.set_xlim(X_MIN, X_MAX + 8)   # Extra space on right for sample labels
ax.set_ylim(-15, n * OFFSET_STEP + 30)
ax.set_xlabel(r"2$\theta$ / degrees (Cu K$\alpha$)", fontsize=12)
ax.set_yticks([])                # Suppress y-axis: absolute intensity is not meaningful
ax.set_title("XRD Diffractograms of Starch Types", fontsize=14,
             fontweight="bold", pad=10)
ax.spines["left"].set_visible(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# --- Legend ---
# Build patches only for polymorph types actually present in the dataset.
# The set comprehension avoids a KeyError if POLYMORPH_COLOURS does not
# contain an entry for a polymorph not in the reference library.
present_polymorphs = {REFERENCE_PEAKS.get(s, {}).get("polymorph")
                      for s in starch_order} - {None}
patches = [mpatches.Patch(fc=POLYMORPH_COLOURS[p], label=p)
           for p in ["A-type", "B-type"]
           if p in present_polymorphs]
ax.legend(handles=patches, title="Polymorph", title_fontsize=9,
          fontsize=8.5, loc="upper left", framealpha=0.85)

ax.text(0.5, -0.06,
        "Reference positions shown (\u25b2). Dashed lines: diagnostic 2\u03b8. Region 5–40\u00b0.",
        transform=ax.transAxes, ha="center", fontsize=8, color="grey")

plt.tight_layout()
if SAVE_FIGURE:
    plt.savefig(FIGURE_NAME, dpi=FIGURE_DPI, bbox_inches="tight")
    print(f"\u2713 Figure saved as {FIGURE_NAME}")
plt.show()

### What to look for in the waterfall plot

**B-type polymorphs** (Potato, Tapioca) show a distinctive peak at low angle (~5–6°).
This arises from the wider spacing of the double-helix network in the hexagonal B-type lattice.

**A-type polymorphs** (Wheat, Corn) lack this low-angle peak and instead show a tight cluster
of three peaks between 15° and 18°, plus a strong peak near 23°.
These arise from the more compact monoclinic A-type unit cell.

> **Student task:** Examine the Unknown pattern. Does it have a peak around 5–6°?
> Does the 15–18° cluster look more like Wheat/Corn or Potato/Tapioca?
> Note your observations before moving to the peak detection section.

## 6 · Automated Peak Detection & Assignment

Peaks are detected from the data using a local-maxima algorithm,  
then matched against the reference library (±0.6° tolerance).

In [ ]:
# ============================================================
# PEAK DETECTION AND ASSIGNMENT
# ============================================================
# Bragg peaks are detected algorithmically using scipy.signal.find_peaks(),
# which identifies local maxima satisfying user-specified constraints.
# Each detected peak is then compared against the reference library to
# determine whether it matches a known polymorph reflection.
#
# The algorithm operates on the normalised (0-100) intensity scale so that
# the height threshold is comparable across all samples regardless of their
# raw counting statistics.

# ── PARAMETERS — modify and re-run ────────────────────────────────────────────────
MIN_HEIGHT_PCT  = 0.12   # Minimum peak height as a fraction of the pattern maximum.
                         # A threshold of 12% suppresses the broad amorphous hump
                         # while retaining all genuine Bragg peaks.
                         # Increase to 0.20 if noise spikes appear in the peak table;
                         # decrease to 0.08 if known peaks are being missed.

MIN_DISTANCE_PT = 15     # Minimum separation between adjacent peaks, in data points.
                         # At a typical step size of 0.02°/point, 15 pts ≈ 0.3°.
                         # Prevents a single broad peak from being split into multiple
                         # detections. Decrease for very sharp, closely spaced peaks.

TOLERANCE_DEG   = 0.6    # Maximum allowed difference (degrees 2θ) between a detected
                         # peak and a reference position for the assignment to be made.
                         # 0.6° accommodates small instrument offsets and natural
                         # peak-position variation without over-assigning.

ANALYSIS_XMIN   = 5.0    # 2θ window for peak detection (degrees).
ANALYSIS_XMAX   = 40.0   # All starch Bragg peaks fall within this range.
# ──────────────────────────────────────────────────────────────────────────────

results = []  # Accumulates one row per detected peak across all samples

for starch, (tt, ct_raw, ct_norm) in processed.items():

    # Restrict to the analysis window
    mask = (tt >= ANALYSIS_XMIN) & (tt <= ANALYSIS_XMAX)
    tt_m, ct_m = tt[mask], ct_norm[mask]

    # --- Detect peaks ---
    # find_peaks returns indices of local maxima where:
    #   intensity > MIN_HEIGHT_PCT × pattern_maximum  (rejects low background wiggles)
    #   separation > MIN_DISTANCE_PT points from nearest other peak  (merges doublets)
    min_h = ct_m.max() * MIN_HEIGHT_PCT
    idx, props = find_peaks(ct_m, height=min_h, distance=MIN_DISTANCE_PT)

    # --- Look up reference positions for this sample ---
    ref      = REFERENCE_PEAKS.get(starch, {})
    ref_pos  = ref.get("peaks", [])       # Empty list for Unknown
    ref_poly = ref.get("polymorph", "unknown")

    # --- Assign each detected peak to the closest reference position ---
    for i in idx:
        p = tt_m[i]   # Detected 2θ position
        h = ct_m[i]   # Normalised intensity at this peak

        if ref_pos:
            # Compute the angular distance to every reference peak
            diffs  = [abs(p - r) for r in ref_pos]
            best_i = int(np.argmin(diffs))

            if diffs[best_i] <= TOLERANCE_DEG:
                # Within tolerance: assign to the nearest reference
                assignment = f"{ref_pos[best_i]:.1f}° ({ref_poly})"
            else:
                # Outside tolerance: peak detected but does not match any reference
                assignment = "unassigned"
        else:
            # Unknown sample: no reference to compare against
            assignment = "unassigned"

        results.append({"Starch":       starch,
                         "Detected_2θ":  round(p, 2),
                         "Rel_Intensity": round(h, 1),
                         "Assignment":   assignment})

# --- Collate and display ---
df_peaks = pd.DataFrame(results).sort_values(["Starch", "Detected_2θ"])
df_peaks.index = range(len(df_peaks))

print("Peak detection & assignment results:")
print(df_peaks.to_string(index=False))

# Save for inclusion in the lab report
df_peaks.to_csv("xrd_peaks_table.csv", index=False)
print("\n\u2713 Saved: xrd_peaks_table.csv")

### Reading the peak table

Each row is a peak detected in the smoothed diffractogram.

| Column | Meaning |
|--------|---------|
| `Starch` | Sample the peak belongs to |
| `Detected_2θ` | Measured 2θ position (degrees) |
| `Rel_Intensity` | Peak height on the 0–100 normalised scale |
| `Assignment` | Closest reference position (±0.6°) and polymorph, or *unassigned* |

> **Student task:** Filter the table rows for `Unknown`.
> Compare the detected 2θ values against the A-type positions (15.3°, 17.1°, 18.0°, 23.0°)
> and B-type positions (5.6°, 14.4°, 17.2°, 19.8°, 22.2°, 24.0°).
> Which set matches more closely?

## 7 · Relative Crystallinity Estimate

Simple Ruland–Vonk approach: integrates counts above a linear amorphous baseline.
> ⚠️ This gives a *relative* estimate only. For absolute crystallinity, use  
> profile-fitting software (TOPAS, HighScore Plus, or GSAS-II).

In [ ]:
# ============================================================
# RELATIVE CRYSTALLINITY ESTIMATE
# ============================================================
# The degree of crystallinity is estimated using the Ruland-Vonk approach:
# the total scattered intensity is partitioned into a crystalline fraction
# (sharp Bragg peaks) and an amorphous fraction (broad diffuse hump).
#
#   Crystallinity (%) = 100 × A_crystalline / A_total
#
# where A_total is the area under the diffractogram and A_crystalline is the
# area above a linear baseline drawn between the pattern endpoints.
# The baseline approximates the slowly varying amorphous scattering contribution.
#
# Important caveats:
#   - This linear baseline is a first-order approximation. More accurate methods
#     (e.g., profile fitting with TOPAS or GSAS-II) use physically motivated
#     amorphous scattering models.
#   - We use the raw (smoothed but NOT normalised) count data for integration.
#     Min-max normalisation would anchor the minimum and maximum to fixed values
#     across samples, which would distort the amorphous/crystalline area ratio.
#   - Typical native starch crystallinity is 20-45 %; values outside this range
#     may indicate gelatinisation, physical damage, or a measurement issue.

# ── PARAMETERS — modify and re-run ────────────────────────────────────────────────
CRYST_XMIN = 5.0    # Start of integration window (degrees).
                    # 5° captures the B-type low-angle peak at 5.6°.
CRYST_XMAX = 35.0   # End of integration window (degrees).
                    # Beyond 35° starch has no significant Bragg peaks,
                    # so extending further only adds amorphous background noise.
# ──────────────────────────────────────────────────────────────────────────────

cryst_results = []

for starch, (tt, ct_raw, ct_norm) in processed.items():

    # Restrict to integration window using raw (smoothed, non-normalised) counts
    mask = (tt >= CRYST_XMIN) & (tt <= CRYST_XMAX)
    tt_m, ct_m = tt[mask], ct_raw[mask]

    if len(tt_m) < 2:
        continue  # Skip if the window is empty (should not happen with real data)

    # --- Construct linear amorphous baseline ---
    # The baseline connects the intensity values at the two ends of the window.
    # np.interp evaluates this straight line at every 2θ point in the window.
    baseline = np.interp(tt_m,
                         [tt_m[0], tt_m[-1]],   # x endpoints
                         [ct_m[0], ct_m[-1]])    # y endpoints (intensities)

    # --- Integrate crystalline and total areas ---
    # np.maximum(..., 0) clips points where the pattern dips below the baseline
    # (an artefact of noise or an imperfect linear fit), preventing negative
    # contributions to the crystalline area.
    crystalline_area = np.trapz(np.maximum(ct_m - baseline, 0), tt_m)
    total_area       = np.trapz(ct_m, tt_m)

    cryst_pct = (round(100 * crystalline_area / total_area, 1)
                 if total_area > 0 else float('nan'))

    poly = REFERENCE_PEAKS.get(starch, {}).get("polymorph", "unknown")
    cryst_results.append({"Starch":          starch,
                           "Polymorph":        poly,
                           "Crystallinity_%":  cryst_pct})

df_cryst = (pd.DataFrame(cryst_results)
              .sort_values("Crystallinity_%", ascending=False))

print("Estimated relative crystallinity (Ruland–Vonk, linear baseline):")
print(df_cryst.to_string(index=False))

# --- Bar chart ---
fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.bar(df_cryst["Starch"],
              df_cryst["Crystallinity_%"],
              color=[STARCH_COLOURS.get(s, "#888") for s in df_cryst["Starch"]],
              edgecolor="white", linewidth=0.8)
ax.bar_label(bars, fmt="%.1f%%", fontsize=9, padding=3)
ax.set_ylabel("Relative crystallinity (%)")
ax.set_title("Estimated Relative Crystallinity by Starch Type", fontweight="bold")
ax.set_ylim(0, df_cryst["Crystallinity_%"].max() * 1.2)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig("xrd_crystallinity.png", dpi=150, bbox_inches="tight")
plt.show()
print("\u2713 Saved: xrd_crystallinity.png")

### Interpreting relative crystallinity

Native starches typically display **20–45 % relative crystallinity** under Cu Kα radiation.
Higher values reflect longer amylopectin chain lengths and more ordered granule packing.
Lower values can result from physical damage (milling), gelatinisation, or
the presence of an amorphous fraction.

> **Student task:** How does the Unknown crystallinity compare to the reference starches?
> Is it within the expected range for a native starch?

## 8 · Unknown Sample Identification

Compare the Unknown sample against the reference library using  
peak position matching score.

In [ ]:
# ============================================================
# UNKNOWN SAMPLE IDENTIFICATION
# ============================================================
# The Unknown sample is compared against each entry in the reference library
# using a recall-based peak-matching score:
#
#   Match score (%) = 100 × (reference peaks detected in Unknown)
#                             / (total reference peaks for that starch)
#
# A reference peak is considered 'detected' if at least one of the Unknown's
# detected peaks falls within MATCH_TOLERANCE degrees of it.
#
# A score of 100 % means every diagnostic peak of the reference starch was
# found in the Unknown; 0 % means none were. Scores ≥ 70 % provide strong
# evidence; 50-70 % is suggestive.
#
# Note: the scoring uses the peak list from Section 6 (same MIN_HEIGHT_PCT
# and MIN_DISTANCE_PT). If the score is unexpectedly low, try reducing
# MIN_HEIGHT_PCT in Section 6 and re-running from there.

# ── PARAMETERS — modify and re-run ────────────────────────────────────────────────
UNKNOWN_LABEL   = "Unknown"  # Label of the unknown sample in spectra_dict.
                              # Change this if your sample has a different label.

MATCH_TOLERANCE = 0.7        # Maximum angular distance (degrees 2θ) for a detected
                              # Unknown peak to count as matching a reference peak.
                              # Slightly wider than TOLERANCE_DEG in Section 6 to
                              # accommodate systematic offsets between the Unknown
                              # measurement run and the reference measurements.
# ──────────────────────────────────────────────────────────────────────────────

if UNKNOWN_LABEL not in processed:
    print(f"No sample labelled '{UNKNOWN_LABEL}' found in the dataset. Skipping.")
else:
    tt, ct_raw, ct_norm = processed[UNKNOWN_LABEL]

    # Restrict to the same analysis window used in Section 6
    mask  = (tt >= ANALYSIS_XMIN) & (tt <= ANALYSIS_XMAX)
    tt_m  = tt[mask]
    ct_m  = ct_norm[mask]

    # --- Detect peaks in the Unknown pattern ---
    # Uses identical algorithm and thresholds to Section 6 for consistency.
    idx, _ = find_peaks(ct_m,
                        height=ct_m.max() * MIN_HEIGHT_PCT,
                        distance=MIN_DISTANCE_PT)
    unk_peaks = tt_m[idx]
    print(f"Detected peaks in '{UNKNOWN_LABEL}': {np.round(unk_peaks, 2).tolist()}")

    # --- Score against each reference starch ---
    scores = []
    for starch, info in REFERENCE_PEAKS.items():

        ref_pos = np.array(info["peaks"])
        n_ref   = len(ref_pos)

        # For each reference peak, check whether any Unknown peak is within tolerance.
        # The outer sum counts how many reference peaks were matched.
        # Using any() rather than np.min avoids breaking when unk_peaks is empty.
        matches = sum(
            any(abs(up - rp) <= MATCH_TOLERANCE for rp in ref_pos)
            for up in unk_peaks
        )

        score = round(100 * matches / n_ref, 1) if n_ref > 0 else 0.0
        scores.append({"Reference":       starch,
                        "Polymorph":       info["polymorph"],
                        "Match_score_%":   score,
                        "Matched_peaks":   matches,
                        "Total_ref_peaks": n_ref})

    df_scores = (pd.DataFrame(scores)
                   .sort_values("Match_score_%", ascending=False)
                   .reset_index(drop=True))

    print("\nMatch scores against reference library:")
    print(df_scores.to_string(index=False))

    best = df_scores.iloc[0]
    print(f"\n\U0001f3c6 Best match: {best['Reference']} ({best['Polymorph']}) "
          f"— {best['Match_score_%']:.0f}% of reference peaks matched")

### Interpreting the identification result

The match score reports the percentage of a reference starch's diagnostic peaks
that were detected in the Unknown pattern (within ±0.7°).
A score ≥ 70 % provides strong evidence; 50–70 % is suggestive.

> **Student task:** Using your peak matching results *and* the waterfall plot,
> answer the following questions in your lab report:
>
> 1. Which starch does the Unknown most closely resemble, and what is its polymorph type
>    (A-type or B-type)?
> 2. How confident are you in the identification? Which evidence is most persuasive?
> 3. What are the practical implications for food fraud?
>    For example, which food commodities (spices, dairy powders, cocoa) could be adulterated
>    with this starch, and why might it be economically attractive?
>
> *Hint:* Starches from different botanical sources command different prices and have
> different functional properties (thickening power, gelatinisation temperature).
> Substituting a cheaper starch for a more expensive ingredient is a well-documented
> form of economically motivated adulteration.

---
## References

- Zobel, H.F. (1988). Molecules to granules: a comprehensive starch review. *Starch*, 40(2), 44–50.  
- Imberty, A. et al. (1991). The double-helical nature of the crystalline part of A-starch. *J. Mol. Biol.*, 201, 365–378.  
- Bogracheva, T.Y. et al. (1998). The effect of mutant genes at the r, rb, rug3, rug4, rug5 and lam loci on the granular structure and physicochemical properties of pea seed starch. *Carbohydr. Polym.*, 37, 323–334.  
- Ruland, W. (1961). X-ray determination of crystallinity and diffuse disorder scattering. *Acta Cryst.*, 14, 1180–1185.

---
*FBMFOR · MSc Food Technology and Quality Assurance · University of Reading*
